# JS/CSS Asset Builder (`dist/`)

> Extracts `#| export_js` and `#| export_css` cells from feature notebooks in `nbs/craft/` into a standalone `dist/` build directory, copying any not-yet-migrated root assets so `dist/` is always complete and verifiable via `--assets-dir dist`.

In [ ]:
#| default_exp build

In [ ]:
#| export
import json, shutil, subprocess
from collections import defaultdict
from fastcore.nbio import read_nb
from fastcore.utils import *
from fastcore.xml import FT, Safe, ft, to_xml
import fastcore.xml as fx

from craft_dev.auth import REPO_ROOT

In [ ]:
#| export
DEFAULT_DIST_DIR = REPO_ROOT/'dist'
DEFAULT_CRAFT_NBS_DIR = REPO_ROOT/'nbs'/'craft'
CRAFT_ASSET_FILES = (
    'CRAFT.css', 'CRAFT.js', 'craft_md.js',
    'hotkey_manager.js', 'hotkey_builtins.js',
    'quercus-treeview.css', 'quercus-treeview.js',
)
_EXPORT_RE = re.compile(r'^#\|\s*export[_-](js|css|ft)\s+(\S+)\s*$')

## Extracting JS & CSS Cells from Notebooks

Feature notebooks mark exported asset cells with `#| export_js <filename>` or `#| export_css <filename>` (paired with `#| eval: false` so `nbdev-test` does not execute raw JS/CSS as Python).

In [ ]:
#| export
def svg_icon(paths, size=15, sw=1.6, **kw):
    "Build an inline SVG `FT` icon from a path string or list of `FT`/path strings."
    kids = [ft('path', d=p) if isinstance(p, str) else p for p in listify(paths)]
    return ft('svg', *kids, aria_hidden='true', viewBox='0 0 24 24', width=str(size), height=str(size),
              fill='none', stroke='currentColor', stroke_width=str(sw),
              stroke_linecap='round', stroke_linejoin='round', **kw)

def ft_templates_js(tpls, var='__craftTpl'):
    "Compile `{name: FT | str}` templates into a compact JS `window[var]` assignment."
    rendered = {k: to_xml(v, indent=False).strip() if isinstance(v, (FT, tuple, list)) else str(v)
                for k, v in tpls.items()}
    return f"window.{var} = Object.assign(window.{var} || {{}}, {json.dumps(rendered, ensure_ascii=False)});"

def ft_config_js(cfg, var='__craftCfg'):
    "Serialize Python config/tokens dict into a compact JS `window[var]` assignment."
    return f"window.{var} = Object.assign(window.{var} || {{}}, {json.dumps(cfg, ensure_ascii=False)});"

def _parse_asset_cell(src):
    lines = src.splitlines()
    kind, target, body = None, None, []
    for ln in lines:
        if m := _EXPORT_RE.match(ln): kind, target = m.group(1), m.group(2)
        elif ln.startswith('#|'): continue
        else: body.append(ln)
    return kind, target, '\n'.join(body).strip()

def _eval_ft_cell(code, ns):
    ns.pop('TEMPLATES', None); ns.pop('CONFIG', None)
    exec(code, ns)
    parts = []
    if tpls := ns.get('TEMPLATES'): parts.append(ft_templates_js(tpls))
    if cfg := ns.get('CONFIG'): parts.append(ft_config_js(cfg))
    return '\n'.join(parts)

def extract_nb_assets(nb_path):
    "Extract `{target_file: [(cell_id, code), ...]}` from `#| export_js`, `#| export_css`, and `#| export_ft` cells."
    nb = read_nb(nb_path)
    out, ft_ns = defaultdict(list), None
    for c in nb.cells:
        if c.cell_type != 'code': continue
        kind, target, code = _parse_asset_cell(c.source)
        if not (target and code): continue
        if kind == 'ft':
            if ft_ns is None:
                ft_ns = {k: getattr(fx, k) for k in dir(fx) if not k.startswith('_')}
                ft_ns.update(L=L, merge=merge, listify=listify, svg_icon=svg_icon, ft_templates_js=ft_templates_js, ft_config_js=ft_config_js)
            code = _eval_ft_cell(code, ft_ns)
        if code: out[target].append((c.get('id', ''), code))
    return dict(out)

In [ ]:
import tempfile
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq

`extract_nb_assets` collects `#| export_js` and `#| export_css` cells in cell order and strips `#|` directive lines.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    nb_p = Path(td)/'demo.ipynb'
    nb = new_nb([
        mk_cell('#| export_js demo.js\n#| eval: false\nconst a = 1;'),
        mk_cell('#| export_css demo.css\n#| eval: false\n.x { color: red; }'),
        mk_cell('x = 1 + 1'),
    ])
    write_nb(nb, nb_p)
    got = extract_nb_assets(nb_p)
    test_eq(sorted(got), ['demo.css', 'demo.js'])
    test_eq(got['demo.js'][0][1], 'const a = 1;')
    test_eq(got['demo.css'][0][1], '.x { color: red; }')

`#| export_ft` evaluates FastHTML / `fastcore.xml` `FT` trees at build time and emits compact HTML/SVG strings into `window.__craftTpl`, ready for `htmx.process(...)` in the browser.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    nb_p = Path(td)/'ft_demo.ipynb'
    nb = new_nb([
        mk_cell('#| export_ft demo.js\nTEMPLATES = dict(btn=Button(svg_icon("M12 6v12"), "Add", type="button", hx_post="/add_below_", hx_include="#kid", hx_swap="none"))'),
    ])
    write_nb(nb, nb_p)
    js = extract_nb_assets(nb_p)['demo.js'][0][1]
    assert 'hx-post=\\"/add_below_\\"' in js and 'viewbox=\\"0 0 24 24\\"' in js and '__craftTpl' in js

## Building `dist/`

`build_dist` merges all feature notebooks in `nbs/craft/` (in sorted filename order), copies any remaining assets from the repo root so `dist/` is always a complete drop-in `--assets-dir`, and syntax-checks every generated `.js` file with `node --check`.

In [ ]:
#| export
def _check_js_syntax(js_path):
    res = subprocess.run(['node', '--check', str(js_path)], capture_output=True, text=True)
    if res.returncode: raise RuntimeError(f"node --check failed for {js_path}:\n{res.stderr}")

def build_dist(
    out_dir=DEFAULT_DIST_DIR, # Output directory for assembled CRAFT assets
    nbs_dir=DEFAULT_CRAFT_NBS_DIR, # Directory of feature notebooks containing `#| export_js`/`export_css` cells
    src_dir=REPO_ROOT, # Repo root used to copy not-yet-migrated assets and `vscode-icons/`
    check_js=True, # Run `node --check` on every `.js` file in `out_dir`
):
    "Assemble a complete CRAFT asset bundle in `out_dir` from feature notebooks plus unmigrated root files."
    out_dir, nbs_dir, src_dir = Path(out_dir), Path(nbs_dir), Path(src_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    chunks = defaultdict(list)
    if nbs_dir.is_dir():
        for nb_p in sorted(nbs_dir.ls(file_exts='.ipynb')):
            rel = nb_p.relative_to(src_dir) if nb_p.is_relative_to(src_dir) else nb_p.name
            for target, items in extract_nb_assets(nb_p).items():
                for cid, code in items:
                    chunks[target].append(f"/* %% {rel} #{cid} */\n{code}")

    report = {}
    for target, parts in sorted(chunks.items()):
        text = '\n\n'.join(parts) + '\n'
        dest = out_dir/target
        dest.mk_write(text)
        report[target] = {'source': 'nb', 'lines': len(text.splitlines())}

    for name in CRAFT_ASSET_FILES:
        if name in report: continue
        src = src_dir/name
        if src.is_file():
            dest = out_dir/name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
            report[name] = {'source': 'copied', 'lines': len(src.read_text(encoding='utf-8').splitlines())}

    icons_src = src_dir/'vscode-icons'
    if icons_src.is_dir():
        shutil.copytree(icons_src, out_dir/'vscode-icons', dirs_exist_ok=True)

    if check_js:
        for target in report:
            if target.endswith('.js'): _check_js_syntax(out_dir/target)
    return report

`build_dist` lets a single feature notebook override one asset while copying the rest into an isolated build directory.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    nb_dir, out_dir = td/'nbs', td/'dist'
    nb_dir.mkdir()
    write_nb(new_nb([mk_cell('#| export_js hotkey_builtins.js\n#| eval: false\nconst _ok = 1;')]), nb_dir/'00_hk.ipynb')
    rep = build_dist(out_dir=out_dir, nbs_dir=nb_dir, src_dir=REPO_ROOT, check_js=True)
    test_eq(rep['hotkey_builtins.js']['source'], 'nb')
    test_eq(rep['CRAFT.js']['source'], 'copied')
    test_eq((out_dir/'CRAFT.css').is_file(), True)

Verify the repository shell installers (`setup.sh`, `zsh-setup.sh`) pass `bash -n`.

In [ ]:
for sh in ('setup.sh', 'zsh-setup.sh'):
    res = subprocess.run(['bash', '-n', str(REPO_ROOT/sh)], capture_output=True, text=True)
    assert res.returncode == 0, f"bash -n failed for {sh}:\n{res.stderr}"